In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud import bigquery

In [ ]:
# =============================================================
# 1. CẤU HÌNH HỆ THỐNG
# =============================================================
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"
TABLE_NAME = "product_user_mapping" # Bạn có thể đổi tên bảng tại đây

# Đường dẫn dữ liệu
MASTER_PATH = f"gs://{BUCKET}/silver-zone/silver-master-table"
TEMP_GCS_BUCKET = BUCKET

# Khởi tạo Spark
spark = DataprocSparkSession.builder.getOrCreate()

Using existing Dataproc Session (configuration changes may not be applied): https://console.cloud.google.com/dataproc/interactive/asia-southeast1/sc-20260404-020349-rzic3e?project=project-7f16ff1d-9026-4799-bfa


In [ ]:
# =============================================================
# 2. HÀM XỬ LÝ CHÍNH
# =============================================================
def build_gold_product_user_mapping():
    print(f" Bắt đầu quy trình chuyển đổi dữ liệu cho bảng: {TABLE_NAME}")

    # Đọc dữ liệu từ Silver Zone
    df_master = spark.read.parquet(MASTER_PATH)

    # ---------------------------------------------------------
    # TRÍCH XUẤT 2 CỘT CẦN THIẾT VÀ LỌC TRÙNG
    # ---------------------------------------------------------
    print(" Đang trích xuất cột parent_asin và user_id...")

    # Dùng .distinct() để đảm bảo mỗi cặp (parent_asin, user_id) là duy nhất
    final_df = df_master.select("parent_asin", "user_id").distinct()

    # ---------------------------------------------------------
    # LƯU TRỮ VÀO BIGQUERY
    # ---------------------------------------------------------
    full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
    print(f" Đang lưu bảng vào BigQuery: {full_table_id}")

    # Chế độ ghi đè ("overwrite"), đổi thành "append" nếu muốn ghi thêm
    final_df.write.format("bigquery") \
        .option("table", full_table_id) \
        .option("temporaryGcsBucket", TEMP_GCS_BUCKET) \
        .mode("overwrite") \
        .save()

    print(" HOÀN TẤT! Dữ liệu đã sẵn sàng.")
    final_df.show(5)

In [ ]:
# =============================================================
# 3. THỰC THI
# =============================================================
if __name__ == "__main__":
    # 1. Kiểm tra/Tạo Dataset
    bq_client = bigquery.Client(project=PROJECT_ID)
    dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

    try:
        bq_client.get_dataset(dataset_ref)
        print(f" Dataset '{DATASET_ID}' đã tồn tại.")
    except Exception:
        print(f" Tạo mới dataset '{DATASET_ID}'...")
        new_ds = bigquery.Dataset(dataset_ref)
        new_ds.location = "asia-southeast1"
        bq_client.create_dataset(new_ds)

    # 2. Chạy tiến trình xử lý
    build_gold_product_user_mapping()

 Dataset 'gold_zone' đã tồn tại.
 Bắt đầu quy trình chuyển đổi dữ liệu cho bảng: product_user_mapping
 Đang trích xuất cột parent_asin và user_id...
 Đang lưu bảng vào BigQuery: project-7f16ff1d-9026-4799-bfa.gold_zone.product_user_mapping


  0%|           0/135 Tasks

 HOÀN TẤT! Dữ liệu đã sẵn sàng.


  0%|           0/135 Tasks

+-----------+--------------------+
|parent_asin|             user_id|
+-----------+--------------------+
| B00CXW17YG|AHBQU6ZNB75CL5G7H...|
| B0BSQYR27R|AGGPZH3BOQZVPI6CI...|
| B09WJGBR1F|AGNFVFNWARYEWHAXJ...|
| B0038R5HFU|AHXJMBVEIVHCULDLY...|
| B00061HIPI|AHQYNAS2CX3W7KETR...|
+-----------+--------------------+
only showing top 5 rows
